# Session 2: Text Preprocessing — Tokenization, Stemming, Lemmatization, Stop Words



## Setup
Import libraries and download the required NLTK corpora.


In [1]:
import nltk
from nltk.tokenize import (word_tokenize, sent_tokenize, WhitespaceTokenizer,
                            WordPunctTokenizer, TreebankWordTokenizer, RegexpTokenizer)
from nltk.stem import PorterStemmer, SnowballStemmer, LancasterStemmer, WordNetLemmatizer
from nltk.corpus import stopwords, wordnet
from nltk import pos_tag

import pandas as pd

# One-time NLTK data downloads (safe to re-run; skips if already present)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('stopwords', quiet=True)

print("NLTK ready.")


NLTK ready.


---
## Section 1: Tokenization using NLTK & Types of Tokenization

### Q1. Word vs. Sentence Tokenization
Given the paragraph below, apply NLTK's `word_tokenize` and `sent_tokenize`. Print both token lists and report the word count and sentence count. Explain, in your own words, the difference between what each function is doing.


In [ ]:
paragraph = (
    "Dr. Ray couldn't believe the results. The model's accuracy jumped to 95.6%! "
    "\"That's incredible,\" she said, before rushing to email her co-author at ray@example.com."
)

words = word_tokenize(paragraph)
sentences = sent_tokenize(paragraph)

print(f"Word tokens ({len(words)}):\n{words}\n")
print(f"Sentence tokens ({len(sentences)}):")
for s in sentences:
    print(" -", s)


### Q2. Comparing NLTK Tokenizer Types
Tokenize the sentence `"I can't believe it's already 3:30 p.m.! Let's grab coffee, shall we?"` with **four different NLTK tokenizers**: `word_tokenize` (Treebank-style), `WhitespaceTokenizer`, `WordPunctTokenizer`, and `TreebankWordTokenizer`. Print each tokenizer's output side by side, and discuss which technique(s) suit (a) quick-and-dirty splitting where speed matters most, and (b) linguistically careful splitting of contractions and punctuation.


In [ ]:
sentence2 = "I can't believe it's already 3:30 p.m.! Let's grab coffee, shall we?"

tok_word = word_tokenize(sentence2)



In [ ]:
tok_whitespace = WhitespaceTokenizer().tokenize(sentence2)


In [ ]:
tok_wordpunct = WordPunctTokenizer().tokenize(sentence2)


In [ ]:
tok_treebank = TreebankWordTokenizer().tokenize(sentence2)


In [ ]:
print(f"word_tokenize        ({len(tok_word):2d} tokens): {tok_word}")
print(f"WhitespaceTokenizer  ({len(tok_whitespace):2d} tokens): {tok_whitespace}")
print(f"WordPunctTokenizer   ({len(tok_wordpunct):2d} tokens): {tok_wordpunct}")
print(f"TreebankWordTokenizer({len(tok_treebank):2d} tokens): {tok_treebank}")


### Q3. Custom Tokenizer with RegexpTokenizer
Build a custom tokenizer using NLTK's `RegexpTokenizer` that extracts **only alphabetic words**, ignoring numbers, punctuation, and symbols, from the text below. Compare the resulting token count to what `word_tokenize` produces on the same text, and explain the trade-off of using a regex-based tokenizer instead of a general-purpose one.


In [ ]:
text3 = "Order #4521 was placed on 2024-05-19 for $129.99 — contact support@shop.com if it's delayed!"

alpha_tokenizer = RegexpTokenizer(r'[A-Za-z]+')
regexp_tokens = alpha_tokenizer.tokenize(text3)
default_tokens = word_tokenize(text3)

print(f"RegexpTokenizer (alphabetic only) -> {len(regexp_tokens)} tokens:\n{regexp_tokens}\n")
print(f"word_tokenize (default)           -> {len(default_tokens)} tokens:\n{default_tokens}")


---
## Section 2: Text Preprocessing — Stemming and Its Types

### Q4. Porter Stemmer
Apply NLTK's `PorterStemmer` to the word list below, and print the original word alongside its stem. Identify at least one example of **over-stemming** and one example of **under-stemming**, if present.


In [ ]:
words_list = ["running", "runner", "easily", "studies", "studying",
              "connection", "connected", "connects", "happiness", "argued"]



In [ ]:
porter = PorterStemmer()
porter_stems = [porter.stem(w) for w in words_list]




In [ ]:
porter_df = pd.DataFrame({'word': words_list, 'porter_stem': porter_stems})
porter_df

### Q5. Comparing Porter, Snowball, and Lancaster Stemmers
Repeat Q4 using NLTK's `SnowballStemmer('english')` and `LancasterStemmer`, on the same word list. Present all three stemmers' outputs in a single comparison table. Which stemmer is the most aggressive, and which is the most conservative? What practical risk comes with an overly aggressive stemmer?


In [ ]:
snowball = SnowballStemmer('english')
lancaster = LancasterStemmer()




In [ ]:
snowball_stems = [snowball.stem(w) for w in words_list]
lancaster_stems = [lancaster.stem(w) for w in words_list]



In [ ]:
stemmer_comparison = pd.DataFrame({
    'word': words_list,
    'porter': porter_stems,
    'snowball': snowball_stems,
    'lancaster': lancaster_stems
})
stemmer_comparison

---
## Section 3: Text Preprocessing — Lemmatization and Its Types

### Q6. WordNet Lemmatizer — With and Without POS Tags
Apply NLTK's `WordNetLemmatizer` to the same word list from Q4, first **without** specifying a POS tag (default: noun), and then **with** the correct POS tag obtained via `nltk.pos_tag`. Print both sets of results side by side. Explain why POS matters for accurate lemmatization.


In [ ]:
lemmatizer = WordNetLemmatizer()

# Lemmatize with the default POS (noun)
lemma_default = [lemmatizer.lemmatize(w) for w in words_list]



In [ ]:
# Get POS tags, then map them to WordNet's POS categories
tagged_words = pos_tag(words_list)

def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN  # default fallback



In [ ]:
lemma_with_pos = [lemmatizer.lemmatize(w, get_wordnet_pos(tag)) for w, tag in tagged_words]

lemma_comparison = pd.DataFrame({
    'word': words_list,
    'pos_tag': [t for _, t in tagged_words],
    'lemma_default_noun': lemma_default,
    'lemma_with_correct_pos': lemma_with_pos
})
lemma_comparison


### Q7. Stemming vs. Lemmatization — Combined Comparison
Place the stemming outputs from Q4/Q5 next to the lemmatization output from Q6 for the same word list, in one combined comparison table. Discuss the trade-offs between stemming and lemmatization.


In [ ]:
combined_comparison = pd.DataFrame({
    'word': words_list,
    'porter_stem': porter_stems,
    'snowball_stem': snowball_stems,
    'lancaster_stem': lancaster_stems,
    'lemma_with_pos': lemma_with_pos
})
combined_comparison


---
## Section 4: Text Preprocessing — Stop Word Handling

### Q8. Default Stopword Removal
Using NLTK's built-in English stopword list, tokenize and remove stopwords from the review below. Report the token count before and after stopword removal, and the percentage reduction.


In [ ]:
review = (
    "This product was not what I expected at all. It is not the best "
    "purchase I have made, and I would not recommend it to anyone."
)

review_tokens = word_tokenize(review.lower())


In [ ]:
stop_words = set(stopwords.words('english'))



In [ ]:
review_no_stopwords = [t for t in review_tokens if t.isalpha() and t not in stop_words]



In [ ]:
pct_reduction = (1 - len(review_no_stopwords) / len(review_tokens)) * 100

print(f"Tokens before stopword removal ({len(review_tokens)}): {review_tokens}")
print(f"\nTokens after stopword removal  ({len(review_no_stopwords)}): {review_no_stopwords}")
print(f"\nPercentage reduction: {pct_reduction:.2f}%")


### Q9. Custom Stopword List — Preserving Negation Words
Build a custom stopword list by starting from NLTK's default English list and removing common negation words (`"not"`, `"no"`, `"nor"`, etc.) from it. Re-run stopword removal on the review from Q8 using this custom list, and compare the results. Explain why blindly removing negation words as "stopwords" can be harmful for tasks like sentiment analysis.


In [ ]:
negation_words = {"not", "no", "nor", "never", "none", "neither"}

custom_stop_words = stop_words - negation_words   # remove negations FROM the stopword list, i.e. keep them

review_custom_filtered = [t for t in review_tokens if t.isalpha() and t not in custom_stop_words]

print(f"Default stopword removal    ({len(review_no_stopwords)} tokens): {review_no_stopwords}")
print(f"Negation-preserving removal ({len(review_custom_filtered)} tokens): {review_custom_filtered}")


### Q10. Full Preprocessing Pipeline
Combine everything from this session into a single pipeline — **tokenize → lowercase → remove stopwords → lemmatize (with POS tagging)** — and apply it to the paragraph below. Print the final cleaned list of tokens, and discuss why stopword removal happens **before** lemmatization rather than after.


In [ ]:
raw_text = (
    "The children were playing happily in the parks while their parents "
    "watched them running and laughing loudly near the old fountains."
)

def preprocess_pipeline(text, stop_word_set):
    # 1. Tokenize
    tokens = word_tokenize(text)
    # 2. Lowercase (and keep alphabetic tokens only)
    tokens = [t.lower() for t in tokens if t.isalpha()]
    # 3. Remove stopwords
    tokens = [t for t in tokens if t not in stop_word_set]
    # 4. Lemmatize with POS tagging
    tagged = pos_tag(tokens)
    lemmatized = [lemmatizer.lemmatize(w, get_wordnet_pos(t)) for w, t in tagged]
    return tokens, lemmatized

tokens_before_lemma, final_tokens = preprocess_pipeline(raw_text, stop_words)

print("Raw text:", raw_text)
print("\nAfter tokenize + lowercase + stopword removal:", tokens_before_lemma)
print("\nFinal cleaned tokens (after lemmatization):     ", final_tokens)
